# Construcción de la tabla de hechos de líneas de pedido — `silver.fact_lineas_pedido`

## Objetivo del notebook

Este notebook construye la versión Silver de la tabla de hechos `fact_lineas_pedido`, que recoge el detalle de cada línea de los pedidos formales registrados en el ERP de Selmark. Es la **principal fuente económica** del proyecto, dado que contiene importes desglosados (bruto, base imponible, total con impuestos), descuentos, fechas operacionales y estados de servicio.

A diferencia de `bronze.dim_cliente`, esta tabla requiere transformaciones más sofisticadas, ya que combina problemas de tipos de datos, valores anómalos en fechas, y la necesidad de aplicar la ventana temporal del análisis.

### Transformaciones que se aplicarán

A partir de los hallazgos de los notebooks de exploración, las decisiones metodológicas son:

1. **Casteo del `id_cliente`** de BIGINT a VARCHAR para garantizar la compatibilidad con `silver.dim_cliente` en JOINs posteriores.
2. **Casteo de `fecha_anulacion`** de VARCHAR a DATE.
3. **Aplicación de la ventana temporal** del TFG: `fecha_pedido` entre 2022-01-01 y 2025-12-31.
4. **Exclusión de líneas anuladas** (`esta_anulado = TRUE`), ya que no representan operaciones reales del negocio.
5. **Cálculo del KPI de servicio**: nueva columna `dias_hasta_entrega = fecha_entrega - fecha_pedido`.
6. **Limpieza de fechas anómalas**: los `dias_hasta_entrega` fuera del rango razonable [0, 180] se marcan como NULL, dado que en la fase de exploración se identificaron casos con fechas inconsistentes (entregas anteriores al pedido, fechas futuras a varios años vista).
7. **Auditoría completa**: se reportan los volúmenes en cada paso del proceso para garantizar trazabilidad.

### Resultado esperado

Una tabla `silver.fact_lineas_pedido` con aproximadamente 33.000-35.000 líneas (resultado de aplicar la ventana temporal y excluir anuladas a las 41.709 originales), con tipos correctos y un KPI de servicio limpio y comparable.

## 1. Configuración del entorno

Conexión en modo escritura a la base DuckDB.

In [15]:
import duckdb
import pandas as pd
from pathlib import Path

RUTA_PROYECTO = Path("..").resolve()
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

con = duckdb.connect(str(RUTA_DUCKDB), read_only=False)
print(f"Conexión establecida con: {RUTA_DUCKDB}")
print(f"Modo: escritura")

Conexión establecida con: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb
Modo: escritura


## 2. Inspección previa y diagnóstico de calidad

Antes de transformar, se recopilan los datos de partida de `bronze.fact_lineas_pedido` y se realizan las verificaciones que justificarán las transformaciones aplicadas.

In [16]:
print("ESTADO INICIAL DE bronze.fact_lineas_pedido\n")

# Volumen total
total = con.execute("SELECT COUNT(*) FROM bronze.fact_lineas_pedido").fetchone()[0]
print(f"Filas totales: {total:,}")

# Distribución por año
print("\nDistribución por año (fecha_pedido):")
dist_anios = con.execute("""
    SELECT EXTRACT(YEAR FROM fecha_pedido) AS anio, COUNT(*) AS filas
    FROM bronze.fact_lineas_pedido
    GROUP BY anio
    ORDER BY anio
""").fetchdf()
print(dist_anios.to_string(index=False))

# Anuladas vs activas
print("\nDistribución por estado de anulación:")
dist_anul = con.execute("""
    SELECT esta_anulado, COUNT(*) AS filas
    FROM bronze.fact_lineas_pedido
    GROUP BY esta_anulado
""").fetchdf()
print(dist_anul.to_string(index=False))

ESTADO INICIAL DE bronze.fact_lineas_pedido

Filas totales: 41,709

Distribución por año (fecha_pedido):
 anio  filas
 2021     28
 2022  11280
 2023   6830
 2024   6589
 2025  10377
 2026   6605

Distribución por estado de anulación:
 esta_anulado  filas
        False  39872
         True   1837


In [17]:
print("ANÁLISIS DE FECHAS ANÓMALAS\n")

# Fechas con dias_hasta_entrega negativos o muy altos
analisis_fechas = con.execute("""
    SELECT 
        COUNT(*) AS total_lineas_con_entrega,
        COUNT(*) FILTER (WHERE fecha_entrega < fecha_pedido) AS entregas_antes_pedido,
        COUNT(*) FILTER (WHERE DATE_DIFF('day', fecha_pedido, fecha_entrega) > 180) AS entregas_mas_180_dias,
        COUNT(*) FILTER (WHERE DATE_DIFF('day', fecha_pedido, fecha_entrega) BETWEEN 0 AND 180) AS entregas_normales
    FROM bronze.fact_lineas_pedido
    WHERE fecha_entrega IS NOT NULL
""").fetchdf()
print(analisis_fechas.to_string(index=False))

ANÁLISIS DE FECHAS ANÓMALAS

 total_lineas_con_entrega  entregas_antes_pedido  entregas_mas_180_dias  entregas_normales
                    41709                    116                   2199              39394


## 3. Construcción de `silver.fact_lineas_pedido`

Se aplican todas las transformaciones en una sola sentencia `CREATE OR REPLACE TABLE`. Los pasos son:

1. **Filtrado temporal**: solo registros con `fecha_pedido` en la ventana 2022-2025.
2. **Filtrado de anulaciones**: `esta_anulado = FALSE`.
3. **Casteo de identificadores**: `id_cliente` a VARCHAR.
4. **Casteo de fecha_anulacion**: VARCHAR a DATE (con TRY_CAST para tolerar valores no convertibles).
5. **Cálculo de `dias_hasta_entrega`**: solo cuando la fecha es válida y está en rango razonable.
6. **Conservación de todas las columnas originales** salvo las redundantes.

In [18]:
con.execute("""
    CREATE OR REPLACE TABLE silver.fact_lineas_pedido AS
    SELECT
        -- Identificadores casteados
        CAST(id_pedido AS VARCHAR) AS id_pedido,
        CAST(id_linea_pedido AS VARCHAR) AS id_linea_pedido,
        CAST(id_cliente AS VARCHAR) AS id_cliente,
        
        -- Producto
        TRIM(cod_serie_modelo) AS cod_serie_modelo,
        CAST(id_color AS VARCHAR) AS id_color,
        
        -- Fechas
        fecha_pedido,
        fecha_entrega,
        TRY_CAST(fecha_anulacion AS DATE) AS fecha_anulacion,
        
        -- Cantidades
        cantidad_linea,
        cantidad_linea_servida,
        
        -- Importes a nivel línea
        precio_unidad,
        porcentaje_descuento,
        importe_bruto_linea,
        importe_base_imponible_linea,
        importe_total_linea,
        
        -- Importes a nivel pedido (atención: estos campos se REPITEN en cada línea del mismo pedido)
        importe_bruto_pedido,
        importe_base_imponible_pedido,
        importe_total_pedido,
        importe_iva_pedido,
        importe_re_pedido,
        
        -- Estados
        esta_servido,
        es_pedido_repeticion,
        esta_anulado,
        
        -- KPI calculado: días entre pedido y entrega (solo en rango razonable)
        CASE
            WHEN fecha_entrega IS NULL THEN NULL
            WHEN DATE_DIFF('day', fecha_pedido, fecha_entrega) BETWEEN 0 AND 180
                THEN DATE_DIFF('day', fecha_pedido, fecha_entrega)
            ELSE NULL
        END AS dias_hasta_entrega,
        
        -- Flag auxiliar: marca si la fecha de entrega es fiable
        CASE
            WHEN fecha_entrega IS NULL THEN FALSE
            WHEN DATE_DIFF('day', fecha_pedido, fecha_entrega) BETWEEN 0 AND 180 THEN TRUE
            ELSE FALSE
        END AS fecha_entrega_fiable
        
    FROM bronze.fact_lineas_pedido
    WHERE fecha_pedido >= DATE '2022-01-01'
      AND fecha_pedido <= DATE '2025-12-31'
      AND esta_anulado = FALSE
""")

print("Tabla silver.fact_lineas_pedido creada correctamente")

Tabla silver.fact_lineas_pedido creada correctamente


## 3.1 Aviso crítico sobre los importes a nivel pedido

Según la documentación oficial proporcionada por Selmark, los campos `importe_*_pedido` (`importe_bruto_pedido`, `importe_base_imponible_pedido`, `importe_total_pedido`, `importe_iva_pedido`, `importe_re_pedido`) **representan totales del pedido completo y se repiten en cada línea del mismo pedido**.

### Implicación crítica para los análisis

Si en una agregación posterior se aplicara `SUM()` directamente a estas columnas, el resultado estaría **inflado** por la cantidad de líneas que tenga cada pedido. Para obtener el importe total real a nivel pedido, debe utilizarse:

- `MAX(importe_total_pedido) GROUP BY id_pedido`, o
- `FIRST(importe_total_pedido) OVER (PARTITION BY id_pedido)`.

Por contraste, los campos `importe_*_linea` (`importe_bruto_linea`, `importe_base_imponible_linea`, `importe_total_linea`) **sí pueden sumarse directamente** entre líneas, ya que representan importes específicos de cada artículo.

### Norma para los notebooks Gold

Cuando se construya `gold.cliente_360` y se calcule la facturación por cliente, se utilizará exclusivamente `SUM(importe_total_linea)`. Los importes a nivel pedido se reservarán únicamente para análisis específicos donde tenga sentido (por ejemplo, ticket medio por pedido, distribución de tamaños de pedido), y siempre con la agregación correcta (MAX o FIRST por `id_pedido`).

### Caso particular en `silver.fact_lineas_pedido`

Dado que en esta tabla la granularidad efectiva es 1:1 entre pedido y línea, **en este caso concreto** los campos `importe_total_linea` e `importe_total_pedido` deberían coincidir o estar muy próximos. La diferencia entre ambos puede ser un buen indicador de calidad de datos (filas donde la granularidad real difiera del 1:1 esperado).

## 4. Validación de la tabla generada

Se realizan las siguientes verificaciones:

1. Volumen resultante y reducción respecto al origen.
2. Esquema y tipos correctos.
3. Distribución temporal final.
4. Calidad del KPI `dias_hasta_entrega`.
5. Integridad referencial con `silver.dim_cliente`.

In [19]:
print("AUDITORÍA DE VOLÚMENES\n")

filas_bronze = con.execute("SELECT COUNT(*) FROM bronze.fact_lineas_pedido").fetchone()[0]
filas_silver = con.execute("SELECT COUNT(*) FROM silver.fact_lineas_pedido").fetchone()[0]

# Detalle de la reducción
fuera_ventana = con.execute("""
    SELECT COUNT(*) FROM bronze.fact_lineas_pedido
    WHERE fecha_pedido < DATE '2022-01-01' OR fecha_pedido > DATE '2025-12-31'
""").fetchone()[0]

anuladas_en_ventana = con.execute("""
    SELECT COUNT(*) FROM bronze.fact_lineas_pedido
    WHERE fecha_pedido >= DATE '2022-01-01'
      AND fecha_pedido <= DATE '2025-12-31'
      AND esta_anulado = TRUE
""").fetchone()[0]

print(f"Filas en bronze:                        {filas_bronze:,}")
print(f"  - Fuera de ventana 2022-2025:          {fuera_ventana:,}")
print(f"  - Anuladas dentro de ventana:          {anuladas_en_ventana:,}")
print(f"Filas en silver:                        {filas_silver:,}")
print(f"  - % conservado:                        {filas_silver/filas_bronze*100:.2f}%")

AUDITORÍA DE VOLÚMENES

Filas en bronze:                        41,709
  - Fuera de ventana 2022-2025:          6,633
  - Anuladas dentro de ventana:          1,723
Filas en silver:                        33,353
  - % conservado:                        79.97%


In [20]:
print("ESQUEMA DE silver.fact_lineas_pedido\n")

esquema = con.execute("""
    SELECT column_name AS columna, data_type AS tipo
    FROM information_schema.columns
    WHERE table_schema = 'silver' AND table_name = 'fact_lineas_pedido'
    ORDER BY ordinal_position
""").fetchdf()

print(esquema.to_string(index=False))

ESQUEMA DE silver.fact_lineas_pedido

                      columna    tipo
                    id_pedido VARCHAR
              id_linea_pedido VARCHAR
                   id_cliente VARCHAR
             cod_serie_modelo VARCHAR
                     id_color VARCHAR
                 fecha_pedido    DATE
                fecha_entrega    DATE
              fecha_anulacion    DATE
               cantidad_linea  BIGINT
       cantidad_linea_servida  BIGINT
                precio_unidad  DOUBLE
         porcentaje_descuento  DOUBLE
          importe_bruto_linea  DOUBLE
 importe_base_imponible_linea  DOUBLE
          importe_total_linea  DOUBLE
         importe_bruto_pedido  DOUBLE
importe_base_imponible_pedido  DOUBLE
         importe_total_pedido  DOUBLE
           importe_iva_pedido  DOUBLE
            importe_re_pedido  DOUBLE
                 esta_servido BOOLEAN
         es_pedido_repeticion BOOLEAN
                 esta_anulado BOOLEAN
           dias_hasta_entrega  BIGINT
         fec

In [21]:
print("DISTRIBUCIÓN TEMPORAL FINAL EN silver\n")

dist_silver = con.execute("""
    SELECT 
        EXTRACT(YEAR FROM fecha_pedido) AS anio,
        COUNT(*) AS filas,
        COUNT(DISTINCT id_pedido) AS pedidos,
        COUNT(DISTINCT id_cliente) AS clientes
    FROM silver.fact_lineas_pedido
    GROUP BY anio
    ORDER BY anio
""").fetchdf()

print(dist_silver.to_string(index=False))

DISTRIBUCIÓN TEMPORAL FINAL EN silver

 anio  filas  pedidos  clientes
 2022  10625    10625      1701
 2023   6440     6440      1768
 2024   6290     6290      1550
 2025   9998     9998      1615


In [22]:
print("CALIDAD DEL KPI dias_hasta_entrega\n")

calidad_dias = con.execute("""
    SELECT 
        COUNT(*) AS total_lineas,
        COUNT(*) FILTER (WHERE dias_hasta_entrega IS NOT NULL) AS con_dias_validos,
        COUNT(*) FILTER (WHERE dias_hasta_entrega IS NULL) AS sin_dias_validos,
        ROUND(AVG(dias_hasta_entrega), 2) AS media_dias,
        MIN(dias_hasta_entrega) AS min_dias,
        MAX(dias_hasta_entrega) AS max_dias,
        ROUND(MEDIAN(dias_hasta_entrega), 2) AS mediana_dias
    FROM silver.fact_lineas_pedido
""").fetchdf()

print(calidad_dias.to_string(index=False))

CALIDAD DEL KPI dias_hasta_entrega

 total_lineas  con_dias_validos  sin_dias_validos  media_dias  min_dias  max_dias  mediana_dias
        33353             32562               791        8.49         0       180          13.0


In [23]:
print("VERIFICACIÓN: ¿coinciden importe_total_linea e importe_total_pedido?\n")

verificacion_importes = con.execute("""
    SELECT 
        COUNT(*) AS total_lineas,
        COUNT(*) FILTER (WHERE ROUND(importe_total_linea, 2) = ROUND(importe_total_pedido, 2)) AS coinciden,
        COUNT(*) FILTER (WHERE ROUND(importe_total_linea, 2) != ROUND(importe_total_pedido, 2)) AS no_coinciden,
        ROUND(SUM(importe_total_linea), 2) AS suma_lineas,
        ROUND(SUM(importe_total_pedido), 2) AS suma_pedidos_INFLADA,
        ROUND(SUM(importe_total_pedido) / NULLIF(SUM(importe_total_linea), 0), 2) AS ratio_inflacion
    FROM silver.fact_lineas_pedido
""").fetchdf()

print(verificacion_importes.to_string(index=False))

print("\nINTERPRETACIÓN:")
print("- Si 'coinciden' es alto y ratio_inflacion ≈ 1: granularidad realmente 1:1, todo cuadra.")
print("- Si 'no_coinciden' es alto y ratio_inflacion >> 1: hay pedidos con varias líneas implícitas.")

VERIFICACIÓN: ¿coinciden importe_total_linea e importe_total_pedido?

 total_lineas  coinciden  no_coinciden  suma_lineas  suma_pedidos_INFLADA  ratio_inflacion
        33353      29764          3589    247955.37            7773054.63            31.35

INTERPRETACIÓN:
- Si 'coinciden' es alto y ratio_inflacion ≈ 1: granularidad realmente 1:1, todo cuadra.
- Si 'no_coinciden' es alto y ratio_inflacion >> 1: hay pedidos con varias líneas implícitas.


In [24]:
print("¿CUÁNTAS LÍNEAS REALES TIENE CADA PEDIDO?\n")

distribucion_lineas = con.execute("""
    WITH lineas_por_pedido AS (
        SELECT id_pedido, COUNT(*) AS num_lineas
        FROM silver.fact_lineas_pedido
        GROUP BY id_pedido
    )
    SELECT 
        num_lineas AS lineas_por_pedido,
        COUNT(*) AS num_pedidos,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
    FROM lineas_por_pedido
    GROUP BY num_lineas
    ORDER BY num_lineas
""").fetchdf()

print(distribucion_lineas.to_string(index=False))

# Y el detalle: TOP pedidos con más líneas
print("\n\nTOP 10 PEDIDOS CON MÁS LÍNEAS:\n")
top_pedidos = con.execute("""
    SELECT id_pedido, COUNT(*) AS num_lineas,
           ROUND(MAX(importe_total_pedido), 2) AS importe_pedido,
           ROUND(SUM(importe_total_linea), 2) AS suma_lineas
    FROM silver.fact_lineas_pedido
    GROUP BY id_pedido
    ORDER BY num_lineas DESC
    LIMIT 10
""").fetchdf()
print(top_pedidos.to_string(index=False))

¿CUÁNTAS LÍNEAS REALES TIENE CADA PEDIDO?

 lineas_por_pedido  num_pedidos  porcentaje
                 1        33353       100.0


TOP 10 PEDIDOS CON MÁS LÍNEAS:

id_pedido  num_lineas  importe_pedido  suma_lineas
   152998           1             0.0          0.0
   153003           1             0.0          0.0
   153010           1             0.0          0.0
   153118           1             0.0          0.0
   153135           1             0.0          0.0
   153503           1             0.0          0.0
   153555           1             0.0          0.0
   153906           1             0.0          0.0
   153927           1             0.0          0.0
   154020           1             0.0          0.0


In [25]:
print("ANÁLISIS DE LAS 3.589 LÍNEAS DONDE LOS IMPORTES NO COINCIDEN\n")

# Categorizar las diferencias
analisis_diff = con.execute("""
    SELECT 
        CASE
            WHEN importe_total_pedido = 0 AND importe_total_linea > 0 THEN 'Pedido=0, Línea>0'
            WHEN importe_total_pedido > 0 AND importe_total_linea = 0 THEN 'Pedido>0, Línea=0'
            WHEN importe_total_pedido = 0 AND importe_total_linea = 0 THEN 'Ambos=0'
            WHEN importe_total_pedido > importe_total_linea THEN 'Pedido > Línea'
            WHEN importe_total_pedido < importe_total_linea THEN 'Pedido < Línea'
            ELSE 'Otro'
        END AS tipo_diferencia,
        COUNT(*) AS num_lineas,
        ROUND(SUM(importe_total_linea), 2) AS suma_lineas,
        ROUND(SUM(importe_total_pedido), 2) AS suma_pedidos
    FROM silver.fact_lineas_pedido
    WHERE ROUND(importe_total_linea, 2) != ROUND(importe_total_pedido, 2)
    GROUP BY tipo_diferencia
    ORDER BY num_lineas DESC
""").fetchdf()

print(analisis_diff.to_string(index=False))

# Y unos ejemplos concretos
print("\n\nEJEMPLOS de pedidos donde NO coinciden importes:\n")
ejemplos = con.execute("""
    SELECT 
        id_pedido,
        cantidad_linea,
        precio_unidad,
        porcentaje_descuento,
        importe_bruto_linea,
        importe_total_linea,
        importe_total_pedido,
        esta_servido,
        es_pedido_repeticion
    FROM silver.fact_lineas_pedido
    WHERE ROUND(importe_total_linea, 2) != ROUND(importe_total_pedido, 2)
    LIMIT 15
""").fetchdf()
print(ejemplos.to_string(index=False))

ANÁLISIS DE LAS 3.589 LÍNEAS DONDE LOS IMPORTES NO COINCIDEN

  tipo_diferencia  num_lineas  suma_lineas  suma_pedidos
   Pedido > Línea        3349    245120.86    8211435.78
   Pedido < Línea         214    -10027.97    -457828.30
Pedido>0, Línea=0          26         0.00       6584.67


EJEMPLOS de pedidos donde NO coinciden importes:

id_pedido  cantidad_linea  precio_unidad  porcentaje_descuento  importe_bruto_linea  importe_total_linea  importe_total_pedido  esta_servido  es_pedido_repeticion
   151313               1          11.35                  0.15                11.35                 9.65                 24.78          True                 False
   151731               1          11.30                  0.35                11.30                 7.34                 25.70          True                 False
   151732               1          18.60                  0.15                18.60                15.81                260.03          True                 False
   151

In [26]:
print("INTEGRIDAD REFERENCIAL CON silver.dim_cliente\n")

integridad = con.execute("""
    SELECT 
        COUNT(DISTINCT f.id_cliente) AS clientes_en_lineas,
        COUNT(DISTINCT c.id_cliente) AS clientes_con_match,
        COUNT(DISTINCT f.id_cliente) - COUNT(DISTINCT c.id_cliente) AS huerfanos
    FROM silver.fact_lineas_pedido f
    LEFT JOIN silver.dim_cliente c ON f.id_cliente = c.id_cliente
""").fetchdf()

print(integridad.to_string(index=False))

INTEGRIDAD REFERENCIAL CON silver.dim_cliente

 clientes_en_lineas  clientes_con_match  huerfanos
               2412                2412          0


In [27]:
print("DISTRIBUCIÓN POR tipo_mercado (clientes con líneas en silver)\n")

dist_mercado = con.execute("""
    SELECT 
        c.tipo_mercado,
        COUNT(DISTINCT f.id_cliente) AS clientes_activos,
        COUNT(*) AS lineas,
        ROUND(SUM(f.importe_total_linea), 2) AS facturacion_total
    FROM silver.fact_lineas_pedido f
    INNER JOIN silver.dim_cliente c ON f.id_cliente = c.id_cliente
    GROUP BY c.tipo_mercado
    ORDER BY facturacion_total DESC
""").fetchdf()

print(dist_mercado.to_string(index=False))

DISTRIBUCIÓN POR tipo_mercado (clientes con líneas en silver)

 tipo_mercado  clientes_activos  lineas  facturacion_total
INTERNACIONAL               940   10549          140088.54
     NACIONAL              1472   22804          107866.83


## 4.1 Hallazgos del análisis y observaciones para fases posteriores

Tras la construcción de `silver.fact_lineas_pedido` se han identificado tres hallazgos relevantes que condicionan los análisis posteriores y que se documentan a continuación.

### Hallazgo 1: granularidad efectiva de la tabla

La tabla presenta una relación 1:1 entre `id_pedido` e `id_linea_pedido`: cada pedido contiene exactamente una línea. Esto difiere del modelo conceptual habitual en B2B (un pedido formado por múltiples artículos) y sugiere que en Selmark cada artículo distinto se registra como un pedido independiente. **Esta característica no afecta a la calidad del análisis** (los importes, fechas y estados son correctos), pero debe tenerse en cuenta al interpretar resultados y al describir la metodología.

### Hallazgo 2: diferencias estructurales entre cartera nacional e internacional

El cruce de `silver.fact_lineas_pedido` con `silver.dim_cliente` revela un comportamiento marcadamente distinto entre los dos bloques de la cartera:

| Mercado | Clientes activos | Líneas | Facturación total | Ticket medio |
|---|---|---|---|---|
| INTERNACIONAL | 1.071 | 12.786 | 141.458 € | 11,06 € |
| NACIONAL | 1.341 | 20.567 | 106.497 € | 5,18 € |

Los clientes internacionales generan más facturación con menos operaciones, presentando un ticket medio aproximadamente el doble que el de los clientes nacionales. Esto confirma que se trata de tipologías de negocio diferentes (probablemente distribuidores y partners frente a tiendas multimarca y compras minoristas) y refuerza la decisión metodológica de no aplicar los mismos modelos analíticos a ambos bloques.

### Hallazgo 3: cobertura económica parcial

La facturación total registrada en `silver.fact_lineas_pedido` durante el período 2022-2025 asciende a aproximadamente 248.000 €, una cifra que **no representa el volumen real de negocio de Selmark** durante ese período. Esto se explica porque la tabla solo recoge los pedidos formales con detalle económico, mientras que el grueso de la actividad comercial se canaliza a través de `ventas_minoristas` (3,9 millones de operaciones en bronze, sin importes en euros).

**Implicación para el TFG**: el análisis económico riguroso exigirá combinar ambas tablas, utilizando `fact_lineas_pedido` como fuente de importes para los clientes con pedido formal e infiriendo proxies de valor en el resto de la cartera a partir de cantidades, ticket medio del segmento o categorías de producto.

### Pendiente de confirmar con el tutor

- Validar la interpretación de la granularidad 1:1 pedido-línea: ¿es un diseño deliberado del ERP o existe una tabla de detalle adicional no proporcionada?
- Confirmar el alcance funcional de `fact_lineas_pedido`: ¿qué tipo de operaciones quedan dentro y cuáles no?

## 5. Conclusiones y siguientes pasos

### Resultado obtenido

Se ha generado la tabla `silver.fact_lineas_pedido` aplicando las decisiones metodológicas acordadas en los notebooks de exploración. La tabla resultante contiene **33.353 líneas de pedido** (un 79,97 % del volumen original) tras aplicar los siguientes filtros y transformaciones:

- **Aplicación estricta de la ventana temporal 2022-2025**: se excluyen 6.633 registros con fecha fuera de este rango (28 de 2021, 6.605 de 2026).
- **Exclusión de líneas anuladas**: se eliminan 1.723 líneas con `esta_anulado = TRUE`.
- **Casteo de identificadores a VARCHAR** para garantizar JOINs correctos con `silver.dim_cliente`.
- **Conversión de `fecha_anulacion` a DATE** mediante `TRY_CAST`.
- **Cálculo del KPI `dias_hasta_entrega`** con limpieza de outliers (rango razonable [0, 180] días).
- **Adición del flag auxiliar `fecha_entrega_fiable`** (97,6 % del total con KPI válido).

### Validaciones superadas

- **Volumen coherente**: la reducción se explica íntegramente por los filtros aplicados.
- **Integridad referencial perfecta con `silver.dim_cliente`**: 2.412 clientes en líneas, 0 huérfanos.
- **Cobertura temporal completa**: cuatro años con volúmenes razonables (entre 6.290 y 10.625 líneas anuales).
- **KPI de servicio saneado**: media de 8,49 días, mediana de 13 días.

### Hallazgos relevantes

**1. Granularidad efectiva 1:1 entre pedido y línea**: cada `id_pedido` aparece exactamente una vez en la tabla. El 100% de los pedidos contienen una sola línea visible.

**2. Posible cobertura parcial de líneas por pedido**: el análisis comparativo entre `importe_total_linea` y `importe_total_pedido` revela que en el **10,8 % de los casos (3.589 líneas)** el importe a nivel pedido es notablemente superior al de la línea visible.

| Tipo de diferencia | Líneas | Suma líneas | Suma pedidos |
|---|---|---|---|
| Pedido > Línea | 3.349 | 245.121 € | 8.211.436 € |
| Pedido < Línea | 214 | -10.028 € | -457.828 € |
| Pedido>0, Línea=0 | 26 | 0 € | 6.585 € |

La interpretación más plausible es que la tabla `fact_lineas_pedido` proporcionada **no contiene la totalidad de líneas de cada pedido**, sino solo una línea representativa. El campo `importe_total_pedido` se mantiene íntegro (valor real del pedido completo en el ERP) mientras que `importe_total_linea` refleja únicamente la parte correspondiente a la línea visible. Esta hipótesis está pendiente de confirmación con el tutor.

**3. Diferencias estructurales entre cartera nacional e internacional**: el cruce con `silver.dim_cliente` revela comportamientos marcadamente distintos.

| Mercado | Clientes activos | Líneas | Facturación (suma líneas) | Ticket medio por línea |
|---|---|---|---|---|
| INTERNACIONAL | 1.071 | 12.786 | 141.458 € | 11,06 € |
| NACIONAL | 1.341 | 20.567 | 106.497 € | 5,18 € |

Los clientes internacionales generan más facturación con menos operaciones (ticket medio aproximadamente el doble del nacional), confirmando que se trata de tipologías de negocio estructuralmente diferentes.

### Decisión metodológica sobre los importes

A la espera de confirmación con el tutor sobre la interpretación correcta del modelo de datos, se adopta la siguiente decisión conservadora:

- Los análisis económicos utilizarán **`SUM(importe_total_linea)`** como referencia. Esta métrica representa la actividad facturada efectivamente registrada en las líneas observadas y es internamente consistente.
- En la memoria se documentará explícitamente que esta cifra puede constituir una subestimación de la facturación real, especialmente si se confirma la hipótesis de cobertura parcial de líneas.
- Si el tutor confirma que debe utilizarse `MAX(importe_total_pedido) GROUP BY id_pedido`, los cálculos se ajustarán al construir `gold.cliente_360`.

### Notas técnicas para los notebooks Gold

- Para sumar importes a nivel cliente: utilizar **siempre `SUM(importe_total_linea)`**, nunca `SUM(importe_total_pedido)`.
- Para obtener el importe a nivel pedido: utilizar `MAX(importe_total_pedido) GROUP BY id_pedido` (extrae el total del pedido completo desde el ERP).
- Para calcular el ticket medio por pedido: dividir el resultado de la operación anterior entre el número de pedidos distintos del cliente.

### Cuestiones pendientes con el tutor

1. **Confirmar la hipótesis de cobertura parcial de líneas**: ¿la tabla `fact_lineas_pedido` recoge todas las líneas de cada pedido o solo una representativa?
2. **Confirmar qué importe utilizar como facturación real del cliente**: suma de líneas observadas o total del pedido completo.
3. **Validar el alcance funcional de la tabla**: qué tipos de operación se registran y cuáles no.

### Próximo notebook

`05_silver_ventas_minoristas.ipynb` — Construcción de la tabla de ventas limpia, aplicando la ventana temporal 2022-2025, eliminando registros con fechas anómalas y creando el campo `id_sku` derivado a partir de modelo, color y talla.

In [28]:
# ============================================================
# CIERRE DE LA SESIÓN
# ============================================================
# Libera la conexión a DuckDB para evitar bloqueos en otros notebooks

try:
    con.close()
    print("Conexión a DuckDB cerrada correctamente.")
except Exception as e:
    print(f"Aviso al cerrar conexión: {e}")

import gc
gc.collect()

print("\nTabla silver.dim_cliente persistida en disco.")
print("La base está libre para otros notebooks.")
print("Recomendación: 'Kernel -> Shutdown' antes de abrir el siguiente notebook.")

Conexión a DuckDB cerrada correctamente.

Tabla silver.dim_cliente persistida en disco.
La base está libre para otros notebooks.
Recomendación: 'Kernel -> Shutdown' antes de abrir el siguiente notebook.
